In [0]:
%sql

CREATE TABLE IF NOT EXISTS retailpulse.bronze.ingestion_audit (
    dataset_name STRING,
    run_id STRING,
    source_rows BIGINT,
    bronze_rows BIGINT,
    status STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    error_message STRING
)
USING DELTA;


In [0]:
# RETAILPULSE - BRONZE INGESTION PIPELINE WITH STAGING
# Run definitions first; test orders only before enabling all nine datasets.

from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid

source_folder = "/Volumes/retailpulse/dev/olist_raw/"
target_schema = "retailpulse.bronze"
audit_table = "retailpulse.bronze.ingestion_audit"

expected_row_counts = {
    "customers": 99441, "orders": 99441, "order_items": 112650,
    "order_payments": 103886, "order_reviews": 99224,
    "products": 32951, "sellers": 3095, "geolocation": 1000163,
    "product_category_name_translation": 71
}

datasets = {
    "customers": {"source_file": "olist_customers_dataset.csv", "target_table": "customers", "primary_key": ["customer_id"]},
    "orders": {"source_file": "olist_orders_dataset.csv", "target_table": "orders", "primary_key": ["order_id"]},
    "order_items": {"source_file": "olist_order_items_dataset.csv", "target_table": "order_items", "primary_key": ["order_id", "order_item_id"]},
    "order_payments": {"source_file": "olist_order_payments_dataset.csv", "target_table": "order_payments", "primary_key": ["order_id", "payment_sequential"]},
    "order_reviews": {"source_file": "olist_order_reviews_dataset.csv", "target_table": "order_reviews", "primary_key": ["review_id", "order_id"]},
    "products": {"source_file": "olist_products_dataset.csv", "target_table": "products", "primary_key": ["product_id"]},
    "sellers": {"source_file": "olist_sellers_dataset.csv", "target_table": "sellers", "primary_key": ["seller_id"]},
    "geolocation": {"source_file": "olist_geolocation_dataset.csv", "target_table": "geolocation", "primary_key": []},
    "product_category_name_translation": {"source_file": "product_category_name_translation.csv", "target_table": "product_category_name_translation", "primary_key": ["product_category_name"]}
}


def log_ingestion_audit(dataset_name, run_id, source_rows, bronze_rows,
                        status, start_time, end_time, error_message):
    audit_record = [(
        dataset_name, run_id, source_rows, bronze_rows, status,
        start_time, end_time, error_message
    )]
    audit_schema = """
        dataset_name STRING, run_id STRING, source_rows BIGINT,
        bronze_rows BIGINT, status STRING, start_time TIMESTAMP,
        end_time TIMESTAMP, error_message STRING
    """
    audit_df = spark.createDataFrame(audit_record, schema=audit_schema)
    (audit_df.write.format("delta").mode("append").saveAsTable(audit_table))


def ingest_dataset(dataset_name, dataset_config):
    run_id = (
        f"{dataset_name}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}_"
        f"{uuid.uuid4().hex[:8]}"
    )
    start_time = datetime.now(timezone.utc)
    source_rows = None
    bronze_rows = None
    bronze_column_count = None
    staging_rows = None

    source_file = dataset_config["source_file"]
    primary_key = dataset_config["primary_key"]
    source_path = source_folder + source_file
    target_table = target_schema + "." + dataset_config["target_table"]
    staging_table = target_schema + ".staging_" + dataset_config["target_table"]

    print(f"\nStarting {dataset_name}; run_id={run_id}")
    print(f"Staging: {staging_table} | Production: {target_table}")

    try:
        # 1. Read source. Preserve the special CSV options for order_reviews.
        reader = (
            spark.read.format("csv")
            .option("header", True)
            .option("inferSchema", False)
        )
        if dataset_name == "order_reviews":
            reader = (
                reader.option("multiLine", True)
                .option("quote", '"')
                .option("escape", '"')
                .option("unescapedQuoteHandling", "STOP_AT_DELIMITER")
            )
        dataframe = reader.load(source_path)

        # 2. Validate source row count.
        source_rows = dataframe.count()
        source_column_count = len(dataframe.columns)
        expected_rows = expected_row_counts.get(dataset_name)
        if expected_rows is not None and source_rows != expected_rows:
            raise ValueError(
                f"Expected {expected_rows} source rows, found {source_rows}"
            )

        # 3. Validate source primary key columns, nulls, and uniqueness.
        for column in primary_key:
            if column not in dataframe.columns:
                raise ValueError(f"Primary-key column missing: {column}")
            null_count = dataframe.filter(F.col(column).isNull()).count()
            if null_count:
                raise ValueError(f"{column} has {null_count} NULL values")

        duplicate_groups = 0
        if primary_key:
            duplicate_groups = (
                dataframe.groupBy(*primary_key).count()
                .filter(F.col("count") > 1).count()
            )
        if duplicate_groups:
            raise ValueError(f"Found {duplicate_groups} duplicate key groups")

        # 4. Add and validate ingestion metadata.
        dataframe = dataframe.withColumns({
            "source_file": F.lit(source_file),
            "ingested_at": F.current_timestamp(),
            "run_id": F.lit(run_id)
        })
        metadata_columns = ["source_file", "ingested_at", "run_id"]
        metadata_null_counts = dataframe.agg(*[
            F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
            for c in metadata_columns
        ]).first().asDict()
        if not all(value == 0 for value in metadata_null_counts.values()):
            raise ValueError(f"Metadata validation failed: {metadata_null_counts}")

        # 5. Write to staging ONLY. Production table is not touched here.
        (
            dataframe.write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(staging_table)
        )
        print(f"Written to staging: {staging_table}")

        # 6. Read back staging and validate before promotion.
        staging_df = spark.table(staging_table)
        staging_rows = staging_df.count()
        staging_column_count = len(staging_df.columns)
        expected_columns = source_column_count + 3

        if staging_rows != source_rows:
            raise ValueError(
                f"Staging row count {staging_rows} != source count {source_rows}"
            )
        if staging_column_count != expected_columns:
            raise ValueError(
                f"Expected {expected_columns} staging columns, found {staging_column_count}"
            )

        for column in primary_key:
            if column not in staging_df.columns:
                raise ValueError(f"Staging primary-key column missing: {column}")
        staging_pk_nulls = sum(
            staging_df.filter(F.col(c).isNull()).count()
            for c in primary_key
        )
        staging_duplicate_groups = 0
        if primary_key:
            staging_duplicate_groups = (
                staging_df.groupBy(*primary_key).count()
                .filter(F.col("count") > 1).count()
            )
        if staging_pk_nulls:
            raise ValueError(f"Staging primary keys contain {staging_pk_nulls} NULL values")
        if staging_duplicate_groups:
            raise ValueError(
                f"Staging has {staging_duplicate_groups} duplicate key groups"
            )

        staging_metadata_nulls = staging_df.agg(*[
            F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
            for c in metadata_columns
        ]).first().asDict()
        if not all(value == 0 for value in staging_metadata_nulls.values()):
            raise ValueError(f"Staging metadata validation failed: {staging_metadata_nulls}")

        print("All staging validation checks passed.")

        # 7. Promote ONLY after all staging validations pass.
        # Delta overwrite is atomic for this individual table write, but a
        # failure after promotion may still need recovery via table history.
        (
            staging_df.write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
        )
        print(f"Promoted staging data to production: {target_table}")

        # 8. Verify production after promotion.
        bronze_df = spark.table(target_table)
        bronze_rows = bronze_df.count()
        bronze_column_count = len(bronze_df.columns)
        if bronze_rows != source_rows:
            raise ValueError(
                f"Production rows {bronze_rows} != source rows {source_rows}"
            )
        if bronze_column_count != expected_columns:
            raise ValueError(
                f"Expected {expected_columns} production columns, found {bronze_column_count}"
            )

        # 9. Write SUCCESS audit.
        end_time = datetime.now(timezone.utc)
        log_ingestion_audit(
            dataset_name, run_id, source_rows, bronze_rows, "SUCCESS",
            start_time, end_time, None
        )

        return {
            "dataset": dataset_name, "source_rows": source_rows,
            "staging_rows": staging_rows, "bronze_rows": bronze_rows,
            "bronze_columns": bronze_column_count, "status": "SUCCESS",
            "run_id": run_id, "error": ""
        }

    except Exception as error:
        error_message = str(error)[:1000]
        end_time = datetime.now(timezone.utc)
        print(f"Ingestion failed for {dataset_name}: {error_message}")

        # Best-effort failure audit; do not hide the original error.
        try:
            log_ingestion_audit(
                dataset_name, run_id, source_rows, bronze_rows, "FAILED",
                start_time, end_time, error_message
            )
        except Exception as audit_error:
            print(f"WARNING: Failure audit could not be written: {str(audit_error)[:500]}")

        return {
            "dataset": dataset_name, "source_rows": source_rows,
            "staging_rows": staging_rows, "bronze_rows": bronze_rows,
            "bronze_columns": bronze_column_count, "status": "FAILED",
            "run_id": run_id, "error": error_message
        }


# ============================================================
# TEST ORDERS ONLY FIRST. Do not run all nine yet.
# ============================================================
#test_result = ingest_dataset("orders", datasets["orders"])
#display(spark.createDataFrame([test_result]))

# Inspect latest orders audit record.
#display(
    #spark.table(audit_table)
    #.filter(F.col("dataset_name") == "orders")
    #.orderBy(F.col("start_time").desc())
#)

# After orders passes, uncomment the block below to run all nine.

ingestion_results = []

for dataset_name, dataset_config in datasets.items():
    ingestion_results.append(
        ingest_dataset(dataset_name, dataset_config)
    )

# Display ingestion results
display(spark.createDataFrame(ingestion_results))

# Count successful and failed datasets
successful_datasets = sum(
    r["status"] == "SUCCESS"
    for r in ingestion_results
)

failed_datasets = sum(
    r["status"] == "FAILED"
    for r in ingestion_results
)

# Print ingestion summary
print(f"Total datasets: {len(datasets)}")
print(f"Successful datasets: {successful_datasets}")
print(f"Failed datasets: {failed_datasets}")


In [0]:
%sql
select * from retailpulse.bronze.ingestion_audit

In [0]:
%sql
DESCRIBE HISTORY retailpulse.bronze.orders;